In [11]:
#data loading and reading csv
import pandas as pd
import numpy as np

races = pd.read_csv('races.csv')
results = pd.read_csv('results.csv')
constructors = pd.read_csv('constructors.csv')
driver = pd.read_csv('drivers.csv', encoding='latin1')

print (races.columns)
print (results.columns)
print (constructors.columns)
print (driver.columns)

Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url'], dtype='object')
Index(['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid',
       'position', 'positionText', 'positionOrder', 'points', 'laps', 'time',
       'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
       'fastestLapSpeed', 'statusId'],
      dtype='object')
Index(['constructorId', 'constructorRef', 'name', 'nationality', 'url',
       'Unnamed: 5'],
      dtype='object')
Index(['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob',
       'nationality', 'url'],
      dtype='object')


In [12]:
#keeping only what is needed as per MVP
races_small = races[['raceId', 'year', 'name', 'date']].copy()
drivers_small = driver[['driverId', 'forename', 'surname', 'nationality']].copy()

constructors_small = constructors[['constructorId', 'name', 'nationality']].copy()

results_small = results[['raceId',
                         'driverId',
                         'constructorId',
                         'grid',
                         'position',
                         'positionText',
                         'points',
                         'statusId']].copy()

In [13]:
#merging all the data to one set
df = results_small.merge(races_small, on='raceId', how='left')
df = df.merge(drivers_small, on='driverId', how='left')
df = df.merge(constructors_small, on='constructorId', how='left')

df = df.rename(columns={
    'name_x': 'race_name',
    'name_y': 'constructor_name',
    'nationality_x': 'driver_nationality',
    'nationality_y': 'constructor_nationality'
})

df.head()
#print (df.columns)

,raceId,driverId,constructorId,grid,position,positionText,points,statusId,year,race_name,date,forename,surname,driver_nationality,constructor_name,constructor_nationality
0,18,1,1,1,1.0,1,10.0,1,2008,Australian Grand Prix,2008-03-16,Lewis,Hamilton,British,McLaren,British
1,18,2,2,5,2.0,2,8.0,1,2008,Australian Grand Prix,2008-03-16,Nick,Heidfeld,German,BMW Sauber,German
2,18,3,3,7,3.0,3,6.0,1,2008,Australian Grand Prix,2008-03-16,Nico,Rosberg,German,Williams,British
3,18,4,4,11,4.0,4,5.0,1,2008,Australian Grand Prix,2008-03-16,Fernando,Alonso,Spanish,Renault,French
4,18,5,1,3,5.0,5,4.0,1,2008,Australian Grand Prix,2008-03-16,Heikki,Kovalainen,Finnish,McLaren,British


In [14]:
#sanity checks
df.shape

(23777, 16)

In [15]:
#missing values check
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(20)

# creating indicator for the missing 10550 position data
df["finished"] = df["position"].notna()

In [16]:
#Duplicate check for data
df.duplicated().sum()


np.int64(0)

In [17]:
#converting date to proper format
df["date"] = pd.to_datetime(df["date"])
# metrics for positions and winners
df["podium"] = df["position"].isin([1, 2, 3])
df["win"] = df["position"] == 1


In [18]:
#Final Data set
df['driver_name'] = df['forename'] + ' ' + df['surname']
df_final = df[[
    "year",
    "date",
    "race_name",
    "driver_name",
    "driver_nationality",
    "constructor_name",
    "constructor_nationality",
    "grid",
    "position",
    "positionText",
    "points",
    "finished",
    "podium",
    "win"
]].copy()
df_final.head()

,year,date,race_name,driver_name,driver_nationality,constructor_name,constructor_nationality,grid,position,positionText,points,finished,podium,win
0,2008,2008-03-16,Australian Grand Prix,Lewis Hamilton,British,McLaren,British,1,1.0,1,10.0,True,True,True
1,2008,2008-03-16,Australian Grand Prix,Nick Heidfeld,German,BMW Sauber,German,5,2.0,2,8.0,True,True,False
2,2008,2008-03-16,Australian Grand Prix,Nico Rosberg,German,Williams,British,7,3.0,3,6.0,True,True,False
3,2008,2008-03-16,Australian Grand Prix,Fernando Alonso,Spanish,Renault,French,11,4.0,4,5.0,True,False,False
4,2008,2008-03-16,Australian Grand Prix,Heikki Kovalainen,Finnish,McLaren,British,3,5.0,5,4.0,True,False,False


In [19]:
#Final sanity check
df_final.shape
df_final.isnull().sum()

,0
year,0
date,0
race_name,0
driver_name,0
driver_nationality,0
constructor_name,0
constructor_nationality,0
grid,0
position,10550
positionText,0


In [20]:
#Saving file in colab
df_final.to_csv("f1_Final_Data.csv", index=False)

In [21]:
from google.colab import files
files.download("f1_Final_Data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>